# SINGLE POLYNOMIAL TRIAL FUNCTION (SPTF)

## **Author:** Ridwan Ademola Ibrahim

**Case Study:** IDA-KI OpenLab Research Bridge, TU Dresden — Middle T-Girder

---

## Model Description

The SPTF framework employs a **single neural network** to represent the structural response of the entire **30 m continuous beam**.

### Trial Function

$$
\phi(x) = \frac{x^2 (x-15)^2 (x-30)}{\text{PHI\_SCALE}}
$$

### Hard Boundary Conditions

The trial function is constructed to satisfy the following boundary conditions exactly:

$$
w(0)=0
$$

$$
w'(0)=0
$$

$$
w(15)=0
$$

$$
w'(15)=0
$$

$$
w(30)=0
$$

### Soft Boundary Condition

The following condition is enforced through the loss function:

$$
w''(30)=0
$$

which corresponds to a **zero bending moment at the pin support**.

---

## Loss Components

The total loss consists of four components:

### 1. Data Loss ($L_{\text{data}}$)

Uses measured slope (Tilt-X) data from:

- $x = 11\,\mathrm{m}$
- $x = 19\,\mathrm{m}$

for all **253 observations**.

---

### 2. Physics Loss ($L_{\text{pde}}$)

The governing beam equation is enforced through

$$
w''''(x,t) + \frac{q(x,t)}{EI} = 0
$$

where independent distributed loads are estimated for each span:

$$
q_{AB}(t)
$$

$$
q_{BC}(t)
$$

for every observation.

---

### 3. Continuity Loss ($L_{\text{continuity}}$)

Continuity across the middle support is enforced through moment continuity

$$
w''(14.7,t) \approx w''(15.3,t)
$$

and shear continuity

$$
w'''(14.7,t) \approx w'''(15.3,t)
$$

ensuring smooth transfer of bending moment and shear force between spans.

---

### 4. Moment Boundary Loss ($L_{\text{moment}}$)

The pin-support condition is enforced as

$$
w''(29.9,t) \approx 0
$$

---

## Input Features

The network utilizes:

- Temporal Fourier features
- Thermal Fourier features

to capture cyclic and temperature-dependent structural behavior.

---

## Validation

Model predictions are validated against the **Finite Element Method (FEM)** baseline at the following uninstrumented locations:

- $x = 5\,\mathrm{m}$
- $x = 10\,\mathrm{m}$
- $x = 20\,\mathrm{m}$
- $x = 25\,\mathrm{m}$

---

## Reproducibility

This notebook contains the complete, executable implementation of the **Single Polynomial Trial Function (SPTF)** framework.

All components required to reproduce the reported results are included:

- Data preprocessing
- Fourier feature generation
- Trial function formulation
- Neural network architecture
- Physics-constrained loss functions
- Training procedures
- Model evaluation
- Visualization and plotting routines

Run the notebook cells sequentially from top to bottom to reproduce all reported results.

---

## Imports, Constants, and Configuration

Core libraries, physical constants (EI, L), Fourier frequencies, validation positions, colour schemes, and output directory.

In [1]:
#Import the necessary libraries
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, json
from datetime import datetime, timedelta

torch.set_default_dtype(torch.float64)

data_wd = "C:\\Users\\ridoc\\OneDrive\\Desktop\\Folders\\SMACCs\\Thesis\\Physics Guided Framework\\Draft\\cleaned_tilt_data.csv"
FEA_SLOPE_CSV = 'outputs/processing/Tiltx/FEM/reconstructed_slopes.csv'
FEA_DEFL_CSV  = 'outputs/processing/Tiltx/FEM/reconstructed_deflections.csv'
OUT = os.path.join(os.path.dirname(data_wd), "Models","outputs","processing","Tiltx","Single_Trial_function_PCNN")

os.makedirs(OUT, exist_ok=True)

#Define the given parameters
L_TOTAL = 30.0; L_SPAN = 15.0
EI_NOM = 37.0e6 * 5.20833e-3; EI_INV = 1.0 / EI_NOM
TILTX_SCALE = 1000.0
FOURIER_FREQS = [1, 2, 4, 8, 16]; N_FOURIER = len(FOURIER_FREQS)

#Construct a scaling factor to normalize the polynomial trial function
_x = np.linspace(0, L_TOTAL, 10000)
_phi = _x**2 * (_x - L_SPAN)**2 * (_x - L_TOTAL)
PHI_SCALE = float(np.max(np.abs(_phi)))

#Establish the validation positions
VAL_POS = [5.0, 10.0, 20.0, 25.0]

#Define dictionary to format the color (CP), label (LP) of the loading phases 
CP = {1: '#2196F3', 2: '#4CAF50', 3: '#FF9800', 4: '#F44336'}
LP = {1: 'Ph1', 2: 'Ph2: +tracks', 3: 'Ph3: +veh4.1t', 4: 'Ph4: +veh9t'}
DC = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
      '#42d4f4', '#f032e6', '#bfef45', '#800000', '#469990']

#Define the reference date, which is the starting date of bridge monitoring
REF_DATE = pd.Timestamp('2024-02-01').date()

## Fourier Feature Encoding

To improve the representation of smooth, cyclic, and temperature-driven structural behaviour, scalar inputs are transformed using **Fourier feature encoding**.

The two scalar inputs are:

- Normalised time, \(t\)
- Standardised temperature, \(T\)

Each scalar is mapped into a higher-dimensional feature space using sinusoidal functions at the frequencies

$$
f \in \{1,\,2,\,4,\,8,\,16\}.
$$

For a generic scalar input \(s\), the Fourier feature mapping is defined as

$$
\gamma(s)=
\Big[
\sin(2\pi \cdot 1 \cdot s),
\cos(2\pi \cdot 1 \cdot s),
\sin(2\pi \cdot 2 \cdot s),
\cos(2\pi \cdot 2 \cdot s),
\sin(2\pi \cdot 4 \cdot s),
\cos(2\pi \cdot 4 \cdot s),
\sin(2\pi \cdot 8 \cdot s),
\cos(2\pi \cdot 8 \cdot s),
\sin(2\pi \cdot 16 \cdot s),
\cos(2\pi \cdot 16 \cdot s)
\Big].
$$

Consequently, the total scalar input are

$$
2 \times 5 = 10
$$

dimensional feature vector consisting of:

- 5 sine features
- 5 cosine features

The encoded vectors for time and temperature are then concatenated and supplied to the neural network.

### Motivation

Standard multilayer perceptrons (MLPs) exhibit **spectral bias**, meaning they tend to learn low-frequency patterns more readily than high-frequency patterns. Fourier feature encoding mitigates this limitation by explicitly providing the network with multi-scale periodic representations of the inputs.

For bridge monitoring applications, this encoding also injects prior physical knowledge that thermal effects and environmental loading exhibit smooth and quasi-periodic behaviour over daily, seasonal, and annual cycles. As a result, the network can more effectively learn temperature-dependent structural responses and long-term temporal trends.

In [2]:
#Define a function to do the Fourier transformation for Time and Temperature
#vals is day of the year and freqs is the frequency [1,2,4,8,16]
def fourier_encode(vals, freqs):
    p = []
    for f in freqs:
        p.append(torch.sin(2*np.pi*f*vals))
        p.append(torch.cos(2*np.pi*f*vals))
    return torch.cat(p, dim=1)

## Trial Function: phi(x) = x^2 (x-15)^2 (x-30) / PHI_SCALE

The trial function is a 5th-degree polynomial whose roots enforce 5 hard BCs:
- Double root at x=0: w(0)=0, w'(0)=0 (clamped left support)
- Double root at x=15: w(15)=0, w'(15)=0 (clamped interior support)
- Single root at x=30: w(30)=0 only (pinned right support)

The zero-moment condition w''(30)=0 cannot be hardcoded (single root provides only w=0), so it is enforced as a soft loss. PHI_SCALE normalises the polynomial by its maximum absolute value.

**Known limitation:** This polynomial has the same shape for both spans, but the fixed-pinned span BC should have an asymmetric shape. This causes systematic bias at x=25m.

## Trial Function

$$
\phi(x)=\frac{x^2(x-15)^2(x-30)}{\text{PHI\_SCALE}}
$$

The trial function is a 5th-degree polynomial whose roots enforce five hard boundary conditions   (clamped left support) :

- Double root at \(x=0\):
  
  $$
  w(0)=0,\qquad w'(0)=0
  $$
  

- Double root at \(x=15\) (clamped interior support):
  
  $$
  w(15)=0,\qquad w'(15)=0
  $$
  

- Single root at \(x=30\)  only (pinned right support) :
  
  $$
  w(30)=0
  $$
  
  

The zero-moment condition

$$
w''(30)=0
$$

cannot be hardcoded (a single root provides only \(w=0\)), so it is enforced as a soft loss.

`PHI_SCALE` normalises the polynomial by its maximum absolute value

### Known Limitation

This polynomial has the same shape for both spans, but the fixed–pinned span boundary conditions should produce an asymmetric shape. This causes systematic bias at

$$
x = 25\,\text{m}.
$$

In [3]:
#Define a function for the trial function
def phi_beam(x):
    return x**2 * (x - L_SPAN)**2 * (x - L_TOTAL) / PHI_SCALE

## Main Neural Network (BeamNet)

4-hidden-layer, 64-neuron **tanh** network.

### Input (25 features)

$$
\underbrace{x}_{1}
+
\underbrace{\text{Fourier}_t}_{10}
+
\underbrace{\text{Fourier}_T}_{10}
+
\underbrace{\text{phase}_{\mathrm{oh}}}_{4}
=
25
\ \text{features}
$$

where:

- \(x\): spatial coordinate (1 feature)
- \(Fourier_t\): Fourier encoding of normalised time (10 features)
- \(Fourier_T\): Fourier encoding of standardised temperature (10 features)
- \(phase_oh\): one-hot phase indicator (4 features)

### Initialisation

**Xavier normal** with gain = 0.3.

Small initial weights keep the NN output near zero at epoch 0, so

$$
w = \phi \cdot \mathrm{NN}
\approx 0,
$$

which corresponds to a physically reasonable initial state.

### Why tanh over ReLU?

The PDE requires computing

$$
\frac{\partial^4 w}{\partial x^4}
$$

through the network. The **tanh** activation function is $$ C^{\infty}, $$ meaning that derivatives of all orders exist and are continuous. This allows smooth and stable computation of higher-order derivatives.

By contrast, ReLU activations produce

$$
\frac{\partial^2}{\partial x^2}\mathrm{ReLU}(x)=0
\quad
\text{(almost everywhere)},
$$ making them unsuitable for fourth-order PDE constraints.

In [4]:
#Define a class for the forward pass: 
    #25 inputs, 
    #4 linear hidden layers with tanh activation functions and 
    #1 linear output layer - 64 neurons per layer.
    #initialize the weights with Xavier normal with a gain of 0.3 and use bias as zeros

class BeamNet(nn.Module):
    def __init__(self):
        super().__init__()
        n_in = 1 + 2*N_FOURIER + 2*N_FOURIER + 4  # 25
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=0.3); nn.init.zeros_(m.bias)
                
    #Note that x means input here - not only spatial cordinate
    def forward(self, x):
        return self.net(x)

## PCNN Solver Class

Manages input normalisation, phase-balanced sampling, collocation points, training loop, and prediction.

### Input Normalisation

- First normalize the time and then Fourier-encode it.

  $$
  t_{\text{norm}}=\frac{\text{day\_index}}{N-1}
  \in [0,1]
  $$

 
- First normalize the Temperature and then Fourier-encode it:

  $$
  T_{\text{norm}}
  =
  \frac{T-\mu_T}{\sigma_T}
  $$

  
- One-hot encoded Phase:

### Phase-Balanced Sampling

$$
\text{weight}_i
=
\frac{1}
{\text{days in phase of observation } i}
$$

followed by normalisation.

Ensures each phase contributes equally to mini-batches.

### 4-Component Loss with Ramped Weights

1. **L_data** (weight = 20): Tilt-x match at sensor locations.

2. **L_pde** (weight ramped from 0.1 to 1.0): Beam equation enforced at collocation points.

3. **L_continuity** (weight ramped from 1 to 20): Moment and shear continuity enforced at

   $$
   x = 15\ \mathrm{m}.
   $$

4. **L_moment** (weight ramped from 1 to 20): Zero-moment condition at the pinned support:

   $$
   w''(30) \approx 0.
   $$


### Trainable Load Parameters

$$ q_{AB}(t) $$ and $$q_{BC}(t) $$

one scalar per day per span, initialised at **5 kN/m**. Trained with higher learning rate

$$
\mathrm{LR}_{q}=0.3
$$

than the neural network weights

$$
\mathrm{LR}_{NN}=0.001.
$$

In [5]:
class CEI_PCNN:
    #Here we initialize. 
    # N is number of observation, phases is [1,2,3,4], T_arr is the temperature
    def __init__(self, N, phases, T_arr):
        self.N = N    #number of observations
        self.xi_AB = torch.linspace(0.5, 14.5, 20)    # 20 linear space between 0.5m and 14.5m in span AB
        self.xi_BC = torch.linspace(15.5, 29.5, 20)   # 20 linear space between 15.5m and 29.5m in span BC
        
        #normalize time
        self.t_norm = torch.arange(N, dtype=torch.float64) / (N - 1)     

        # Put the input temperature in an array and normalize it
        T = np.array(T_arr); self.T_norm = torch.tensor((T - T.mean()) / (T.std() + 1e-10))

        #put the input phase in an array and do a one-hot encoding for it
        pa = np.array(phases); ph_oh = torch.zeros(N, 4)
        for i in range(N): ph_oh[i, pa[i]-1] = 1.0
        self.ph_oh = ph_oh

        #begin weight sampling so that each phase contributes equally to the model regardless of how many samples are present
        w = np.zeros(N)
        for ph in [1,2,3,4]: mask = pa == ph; w[mask] = 1.0 / mask.sum()
        w /= w.sum(); self.sw = torch.tensor(w)
        self._net = None
        
    #define a function for the input
    def _inp(self, x, idx):
        ft = fourier_encode(self.t_norm[idx].view(-1,1), FOURIER_FREQS)  #pass the normalize time into the fourier encoder function to get 10 features
        fT = fourier_encode(self.T_norm[idx].view(-1,1), FOURIER_FREQS) #pass the normalize temp. into the fourier encoder function to get 10 features
        return torch.cat([x, ft, fT, self.ph_oh[idx]], dim=1) #return x, 10 t, 10 T, and 4 phases where only one is activated at a time

    #define the solve function that takes the measurement at the two sensor position. 
    #We pass the data in batches of 32. 
    #Use 5000 adam optimization and 80 L-BFGS 
    def solve(self, m11, m19, n_adam=5000, n_lbfgs=80, bs=32):
        net = BeamNet()                        #Call the NN
        q_AB = nn.Parameter(torch.full((self.N,), 5.0))  #learn q_AB as a parameter and initialize at 5.0
        q_BC = nn.Parameter(torch.full((self.N,), 5.0))  #learn q_BC as a parameter and initialize at 5.0
        
        s11 = torch.tensor(m11); s19 = torch.tensor(m19)     #turn the measured column of tiltx11m and tiltx19m to a tensor
        
        opt = torch.optim.Adam([{'params': net.parameters(), 'lr': 1e-3},
                                 {'params': [q_AB, q_BC], 'lr': 3e-1}])     #specify a different learning rate for the NN parameters and q parameters
        
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_adam, eta_min=1e-6)   #use cosine annealing to vary the learning rate with epoch
        
        hist = {'data': [], 'pde': [], 'cont': [], 'mom': [], 'total': []} #empty list to store the progress during training
        
        nc_per = 8    #number of collocations for PDE calculation

        for ep in range(n_adam):
            #Get the index of random batch size (32) observations and clear gradient 
            idx = torch.multinomial(self.sw, bs, replacement=True); opt.zero_grad()

            
            # Data
            x1 = torch.full((bs,1), 11.0).requires_grad_(True)               #create a tensor of spatial position at sensor position (i.e x =11m) of shape batch size
            i1 = self._inp(x1, idx)                                          #put this x1 position in the input function to prepare features
            w1 = phi_beam(x1) * net(torch.cat([x1, i1[:,1:]], 1))            #pass the features to the NN and multiply NN output with trial function to get deflection
            dw1 = torch.autograd.grad(w1.sum(), x1, create_graph=True)[0]    #do the first gradient which is slope 

            
            x2 = torch.full((bs,1), 19.0).requires_grad_(True)               #create a tensor of spatial position at sensor position (i.e x =19m) of shape batch size
            i2 = self._inp(x2, idx)                                          #put this x2 position in the input function to prepare features
            w2 = phi_beam(x2) * net(torch.cat([x2, i2[:,1:]], 1))            #pass the features to the NN and multiply NN output with trial function to get deflection
            dw2 = torch.autograd.grad(w2.sum(), x2, create_graph=True)[0]    #do the first gradient which is slope
            
            Ld = torch.mean((-dw1.squeeze()*1000 - s11[idx])**2) + \
                 torch.mean((-dw2.squeeze()*1000 - s19[idx])**2)            #compute data loss (x=11m + x=19m)
            
            
            # PDE
            idx_c = idx.repeat_interleave(nc_per)                          #repeat the index for nc_per times (i.e 8 times)
            o = torch.ones(bs*nc_per, 1)                                   #create a tensor of ones of batch size x nc_perx1 (i.e 256x1)
            ci = torch.randint(0, 20, (bs*nc_per,))                        #sample random 256 numbers between 0 and 19
            xc = self.xi_AB[ci].view(-1,1).requires_grad_(True)            #use the ci as index to select 256 collocation points in Span AB
            ic = self._inp(xc, idx_c)                                      #create input features from these 256 collocation points
            wc = phi_beam(xc) * net(torch.cat([xc, ic[:,1:]], 1))          #multiply NN output (deflection) with trial function
            
            d1 = torch.autograd.grad(wc, xc, o, create_graph=True)[0]      #first derivative (slope)
            d2 = torch.autograd.grad(d1, xc, o, create_graph=True)[0]      #second derivative (moment)
            d3 = torch.autograd.grad(d2, xc, o, create_graph=True)[0]      #third derivative (shear force)
            d4 = torch.autograd.grad(d3, xc, o, create_graph=True)[0]      #fourth derivative (load)
            
            Lpa = torch.mean((d4 + q_AB[idx].repeat_interleave(nc_per).view(-1,1)*EI_INV).squeeze()**2)  #PDE residual for Span AB

            
            ci2 = torch.randint(0, 20, (bs*nc_per,))                       #sample random 256 numbers between 0 and 19
            xc2 = self.xi_BC[ci2].view(-1,1).requires_grad_(True)          #use the ci as index to select 256 collocation points in Span BC
            ic2 = self._inp(xc2, idx_c)                                    #create input features from these 256 collocation points             
            wc2 = phi_beam(xc2) * net(torch.cat([xc2, ic2[:,1:]], 1))      #multiply NN output (deflection) with trial function
            
            d1b = torch.autograd.grad(wc2, xc2, o, create_graph=True)[0]   #first derivative (slope)
            d2b = torch.autograd.grad(d1b, xc2, o, create_graph=True)[0]   #second derivative (moment)
            d3b = torch.autograd.grad(d2b, xc2, o, create_graph=True)[0]   #third derivative (shear force)
            d4b = torch.autograd.grad(d3b, xc2, o, create_graph=True)[0]   #fourth derivative (load)
            
            Lpb = torch.mean((d4b + q_BC[idx].repeat_interleave(nc_per).view(-1,1)*EI_INV).squeeze()**2)  #PDE residual for Span BC
            
            Lp = Lpa + Lpb                                                  #Add both residuals
            
            # Continuity
            o1 = torch.ones(bs, 1)                                          #create a tensor of ones of shape batch size x 1 (i.e., 256x1)
            xL = torch.full((bs,1), 14.7).requires_grad_(True)              #create a tensor of spatial position near mid support (i.e x =14.7m) of shape batch size 
            iL = self._inp(xL, idx)                                         #put this xL position in the input function to prepare features
            wL = phi_beam(xL) * net(torch.cat([xL, iL[:,1:]], 1))           #pass the features to the NN and multiply the output with trial function
            d1L = torch.autograd.grad(wL, xL, o1, create_graph=True)[0]     #first derivative (slope)
            d2L = torch.autograd.grad(d1L, xL, o1, create_graph=True)[0]    #second derivative (moment)
            d3L = torch.autograd.grad(d2L, xL, o1, create_graph=True)[0]    #third derivative (shear force)
            
            xR = torch.full((bs,1), 15.3).requires_grad_(True)              #create a tensor of spatial position near mid support (i.e x =15.3m) of shape batch size
            iR = self._inp(xR, idx)                                         #put this xR position in the input function to prepare features
            wR = phi_beam(xR) * net(torch.cat([xR, iR[:,1:]], 1))           #pass the features to the NN and multiply the output with trial function
            d1R = torch.autograd.grad(wR, xR, o1, create_graph=True)[0]     #first derivative (slope)
            d2R = torch.autograd.grad(d1R, xR, o1, create_graph=True)[0]    #second derivative (moment)
            d3R = torch.autograd.grad(d2R, xR, o1, create_graph=True)[0]    #third derivative (shear force)
            Lc = torch.mean((d2L.squeeze()-d2R.squeeze())**2) + torch.mean((d3L.squeeze()-d3R.squeeze())**2)  #Moment and shear force loss function component
            
            # Moment BC
            x30 = torch.full((bs,1), 29.9).requires_grad_(True)              #create a tensor of spatial position at pinned support (i.e x =29.9m) of shape batch size
            i30 = self._inp(x30, idx)                                            #put this x30 position in the input function to prepare features
            w30 = phi_beam(x30) * net(torch.cat([x30, i30[:,1:]], 1))            #pass the features to the NN and multiply the output with trial function
            d1_30 = torch.autograd.grad(w30, x30, o1, create_graph=True)[0]      #first derivative (slope)
            d2_30 = torch.autograd.grad(d1_30, x30, o1, create_graph=True)[0]    #second derivative (moment)
            Lm = torch.mean(d2_30.squeeze()**2)

            
            #Loss 
            lp = min(0.1 + ep / n_adam, 1.0)                                   #PDE loss coefficient
            lc = min(1 + ep*20/n_adam, 20.0)                                   #Continuity and Moment loss coefficient
            loss = 20*Ld + lp*Lp + lc*Lc + lc*Lm                               #Total loss function. Note that the data loss coefficient is 20
            
            loss.backward()                                                    #backpropagation
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)              #clip the gradient to avoid excessive gradient. Clip it to 5.0
            opt.step()                                                         #parameter update
            sched.step()                                                       #learning rate update
            hist['data'].append(Ld.item()); hist['pde'].append(Lp.item())      #record data loss and pde loss during training in the empty list we cretaed above
            hist['cont'].append(Lc.item()); hist['mom'].append(Lm.item())      #record continuity and moment loss in the empty list we created above
            hist['total'].append(loss.item())                                  #record total loss in the empty list we created above

            #Print result on screen at every 1000 epochs interval
            if ep % 1000 == 0:
                print(f"    ep {ep}: Ld={Ld.item():.3e} Lp={Lp.item():.3e} Lc={Lc.item():.3e} Lm={Lm.item():.3e}")

        # L-BFGS
        print(f"    L-BFGS ({n_lbfgs} steps)...")
        lbfgs = torch.optim.LBFGS(list(net.parameters())+[q_AB,q_BC], lr=0.3, max_iter=20,
                                    line_search_fn='strong_wolfe')        #create the optimizer for both NN and q(t) parameters
        #begin the iteration (same process as Adam)
        for step in range(n_lbfgs):
            def closure():
                lbfgs.zero_grad()            #clear existing gradient
                bs2 = min(64, self.N)        #create a batch of 64 observations....self.N is 253 
                idx2 = torch.multinomial(self.sw, bs2, replacement=True)  #get the index of the 64 randomly sampled weights
                x1=torch.full((bs2,1),11.0).requires_grad_(True)   #make a 2D matrix of the spatial position at x=11m
                w1=phi_beam(x1)*net(torch.cat([x1,self._inp(x1,idx2)[:,1:]],1))  #get the deflection as usual
                dw1=torch.autograd.grad(w1.sum(),x1,create_graph=True)[0]        #slope
                
                x2=torch.full((bs2,1),19.0).requires_grad_(True)                 #same thing at x=19m
                w2=phi_beam(x2)*net(torch.cat([x2,self._inp(x2,idx2)[:,1:]],1))
                dw2=torch.autograd.grad(w2.sum(),x2,create_graph=True)[0]

                
                Ld=torch.mean((-dw1.squeeze()*1000-s11[idx2])**2)+torch.mean((-dw2.squeeze()*1000-s19[idx2])**2)   #data loss 
                idx_c2=idx2.repeat_interleave(8); o=torch.ones(bs2*8,1)
                xca=self.xi_AB[torch.randint(0,20,(bs2*8,))].view(-1,1).requires_grad_(True)
                wca=phi_beam(xca)*net(torch.cat([xca,self._inp(xca,idx_c2)[:,1:]],1))
                d1=torch.autograd.grad(wca,xca,o,create_graph=True)[0]; d2=torch.autograd.grad(d1,xca,o,create_graph=True)[0]
                d3=torch.autograd.grad(d2,xca,o,create_graph=True)[0]; d4=torch.autograd.grad(d3,xca,o,create_graph=True)[0]
                Lpa=torch.mean((d4+q_AB[idx2].repeat_interleave(8).view(-1,1)*EI_INV).squeeze()**2)
                xcb=self.xi_BC[torch.randint(0,20,(bs2*8,))].view(-1,1).requires_grad_(True)
                wcb=phi_beam(xcb)*net(torch.cat([xcb,self._inp(xcb,idx_c2)[:,1:]],1))
                d1b=torch.autograd.grad(wcb,xcb,o,create_graph=True)[0]; d2b=torch.autograd.grad(d1b,xcb,o,create_graph=True)[0]
                d3b=torch.autograd.grad(d2b,xcb,o,create_graph=True)[0]; d4b=torch.autograd.grad(d3b,xcb,o,create_graph=True)[0]
                Lpb=torch.mean((d4b+q_BC[idx2].repeat_interleave(8).view(-1,1)*EI_INV).squeeze()**2)
                o1=torch.ones(bs2,1)
                xL=torch.full((bs2,1),14.7).requires_grad_(True)
                wL=phi_beam(xL)*net(torch.cat([xL,self._inp(xL,idx2)[:,1:]],1))
                d1L=torch.autograd.grad(wL,xL,o1,create_graph=True)[0]; d2L=torch.autograd.grad(d1L,xL,o1,create_graph=True)[0]
                d3L=torch.autograd.grad(d2L,xL,o1,create_graph=True)[0]
                xR=torch.full((bs2,1),15.3).requires_grad_(True)
                wR=phi_beam(xR)*net(torch.cat([xR,self._inp(xR,idx2)[:,1:]],1))
                d1R=torch.autograd.grad(wR,xR,o1,create_graph=True)[0]; d2R=torch.autograd.grad(d1R,xR,o1,create_graph=True)[0]
                d3R=torch.autograd.grad(d2R,xR,o1,create_graph=True)[0]
                Lc=torch.mean((d2L.squeeze()-d2R.squeeze())**2)+torch.mean((d3L.squeeze()-d3R.squeeze())**2)
                x30=torch.full((bs2,1),29.9).requires_grad_(True)
                w30=phi_beam(x30)*net(torch.cat([x30,self._inp(x30,idx2)[:,1:]],1))
                d1_30=torch.autograd.grad(w30,x30,o1,create_graph=True)[0]
                d2_30=torch.autograd.grad(d1_30,x30,o1,create_graph=True)[0]
                Lm=torch.mean(d2_30.squeeze()**2)
                loss=20*Ld+Lpa+Lpb+20*Lc+20*Lm; loss.backward(); return loss
            try: lbfgs.step(closure)
            except: break
            if step % 20 == 0: print(f"    L-BFGS {step}")

        self._net = net; self._q_AB = q_AB; self._q_BC = q_BC
        return hist
        
    #Make predictions from the Model
    def predict_at(self, positions):
        net = self._net; N = self.N; P = len(positions)      #call the neural network, number of observations, and number of points to reconstruct
        slopes = np.zeros((N, P)); defls = np.zeros((N, P))  #create a 2D tensor (N x P) to store the slope and deflection. Right now, we will leave them at zero and update as we proceed
        idx_all = torch.arange(N)                            #make index for each observation i.e from 0 to 252
        
        for p, xv in enumerate(positions):                                #for index and values in position. Remember position is a list e.g [5,10,20,25]
            x_t = torch.full((N,1), float(xv)).requires_grad_(True)       #create a 2D matrix (N x 1) of the values of position
            inp = self._inp(x_t, idx_all)                                 #produce all features 
            w = phi_beam(x_t) * net(torch.cat([x_t, inp[:,1:]], 1))       # make predictions for deflection i.e trial function x NN output
            dw = torch.autograd.grad(w.sum(), x_t)[0]                     #first derivative to obtain slope
            slopes[:, p] = (-dw.squeeze()*TILTX_SCALE).detach().numpy()   #put the slopes in the empty slopes matrix we created above
            defls[:, p] = (-w.squeeze()*1000).detach().numpy()            #put the deflections in the empty defls matrix we created above   
        return slopes, defls                                              #return the matrix for slope and delfection at the selected position of choice

## Accuracy Metrics

The following metrics are used to evaluate the agreement between model predictions and reference values:

- **Coefficient of Determination (\(R^2\))**
- **Root Mean Square Error (RMSE)**
- **Mean Absolute Error (MAE)**
- **Mean Absolute Percentage Error (MAPE)**
- **Bias**
- **Pearson Correlation Coefficient (\(r\))**

### Notes

- **MAPE** is omitted when the reference values are close to zero to avoid numerical instability and misleading percentage errors.
- **Pearson \(r\)** measures the strength of the linear relationship between predicted and reference values and captures shape agreement independently of offset and scale.

In [6]:
def met(y, yp):                         #y is true value and yp is predicted value
    err = yp - y                        #absolute error between true and predicted value
    ss = np.sum(err**2)                 #sum of the squares of the error
    st = np.sum((y-y.mean())**2)        #sum of the square of the true value and the mean of the true value
    mask = np.abs(y) > 1e-6             #pick only the absolute of the true values greater than 1e-6
    mape = float(np.mean(np.abs(err[mask]/y[mask]))*100) if mask.sum() > 0 else float('nan')   #calculate mean absolute percentage error if the sum of mask is not negative

    #return R2, RMSE,MAE, MAPE, Bias, and r 
    return {'R2': float(1-ss/(st+1e-12)), 'RMSE': float(np.sqrt(np.mean(err**2))),
            'MAE': float(np.mean(np.abs(err))), 'MAPE': mape,
            'Bias': float(np.mean(err)),
            'r': float(np.corrcoef(y.ravel(), yp.ravel())[0,1]) if y.ravel().std()>1e-10 else 0.0}

## Physics Compliance Evaluation

Evaluates physics consistency after training:

- **BC Violations (RMS over all \(N\) observations):** Hard BCs should be approximately 1e-10. The soft BC

  $$
  w''(30) \approx 0
  $$

  depends on optimization quality.

- **Continuity at Interior Support:** Moment and shear jumps should be close to zero:

  $$
  w''(14.7)\approx w''(15.3), \qquad
  w'''(14.7)\approx w'''(15.3).
  $$

- **PDE Residual:** Evaluates

  $$
  w'''' + \frac{q}{EI} = 0,
  $$

  at collocation points. Small residuals indicate a physically consistent beam solution.

**Note:** RMS is used instead of the mean because it penalizes large individual violations more heavily.

In [7]:
def evaluate_physics(solver, N):
    net = solver._net; idx_all = torch.arange(N); results = {}
    bc_v = {}
    for xv, bc in [(0.0,'w(0)'), (15.0,'w(15)'), (30.0,'w(30)')]:
        x_t = torch.full((N,1), xv).requires_grad_(True)
        inp = solver._inp(x_t, idx_all)
        w = phi_beam(x_t) * net(torch.cat([x_t, inp[:,1:]], 1))
        bc_v[bc] = float(torch.sqrt(torch.mean(w**2)).item())
    for xv, bc in [(0.0,"w'(0)"), (15.0,"w'(15)")]:
        x_t = torch.full((N,1), xv).requires_grad_(True)
        inp = solver._inp(x_t, idx_all)
        w = phi_beam(x_t) * net(torch.cat([x_t, inp[:,1:]], 1))
        dw = torch.autograd.grad(w.sum(), x_t)[0]
        bc_v[bc] = float(torch.sqrt(torch.mean(dw**2)).item())
    x30 = torch.full((N,1), 29.95).requires_grad_(True)
    inp30 = solver._inp(x30, idx_all)
    w30 = phi_beam(x30) * net(torch.cat([x30, inp30[:,1:]], 1))
    d1 = torch.autograd.grad(w30.sum(), x30, create_graph=True)[0]
    d2 = torch.autograd.grad(d1.sum(), x30)[0]
    bc_v["w''(30)"] = float(torch.sqrt(torch.mean(d2**2)).item())
    results['bc_violations_rms'] = bc_v

    o1 = torch.ones(N, 1)
    xL = torch.full((N,1), 14.7).requires_grad_(True)
    iL = solver._inp(xL, idx_all)
    wL = phi_beam(xL)*net(torch.cat([xL, iL[:,1:]], 1))
    d1L = torch.autograd.grad(wL, xL, o1, create_graph=True)[0]
    d2L = torch.autograd.grad(d1L, xL, o1, create_graph=True)[0]
    d3L = torch.autograd.grad(d2L, xL, o1)[0]
    xR = torch.full((N,1), 15.3).requires_grad_(True)
    iR = solver._inp(xR, idx_all)
    wR = phi_beam(xR)*net(torch.cat([xR, iR[:,1:]], 1))
    d1R = torch.autograd.grad(wR, xR, o1, create_graph=True)[0]
    d2R = torch.autograd.grad(d1R, xR, o1, create_graph=True)[0]
    d3R = torch.autograd.grad(d2R, xR, o1)[0]
    mj = d2L.squeeze().detach().numpy() - d2R.squeeze().detach().numpy()
    sj = d3L.squeeze().detach().numpy() - d3R.squeeze().detach().numpy()
    results['continuity'] = {'moment_rms': float(np.sqrt(np.mean(mj**2))), 'moment_max': float(np.max(np.abs(mj))),
                              'shear_rms': float(np.sqrt(np.mean(sj**2))), 'shear_max': float(np.max(np.abs(sj)))}

    pde = {}
    for sn, xi_ev, qp in [('AB', torch.linspace(1,14,20), solver._q_AB),
                            ('BC', torch.linspace(16,29,20), solver._q_BC)]:
        res_all = []
        for i in range(N):
            x_t = xi_ev.view(-1,1).clone().requires_grad_(True)
            idx_i = torch.full((20,), i, dtype=torch.long)
            inp = solver._inp(x_t, idx_i)
            ww = phi_beam(x_t)*net(torch.cat([x_t, inp[:,1:]], 1))
            oo = torch.ones_like(ww)
            d1 = torch.autograd.grad(ww, x_t, oo, create_graph=True)[0]
            d2 = torch.autograd.grad(d1, x_t, oo, create_graph=True)[0]
            d3 = torch.autograd.grad(d2, x_t, oo, create_graph=True)[0]
            d4 = torch.autograd.grad(d3, x_t, oo)[0]
            r = d4.squeeze().detach().numpy() + qp[i].detach().item()*EI_INV
            res_all.append(np.mean(r**2))
        pde[sn] = {'rms': float(np.sqrt(np.mean(res_all))), 'max': float(np.sqrt(np.max(res_all)))}
    results['pde_residual'] = pde
    return results

## Helper Functions

Date formatting and phase-transition vertical line markers for plots.

In [8]:
def day_to_date(d): return (pd.Timestamp('2024-02-01') + timedelta(days=int(d))).strftime('%B %d')
def _vlines(ax):
    for d in [21,46,81,148]: ax.axvline(d, color='gray', ls='--', lw=0.8, alpha=0.5)

## Data Loading

Loads cleaned tilt data CSV, computes daily averages, assigns loading phases, and merges with FEM baseline slopes/deflections at all spatial positions (0.5m intervals) for validation. Temperature T = mean of front/back thermocouples at x=11m.

In [9]:
def load_data():
    raw = pd.read_csv(data_wd); raw['Timestamp'] = pd.to_datetime(raw['Timestamp'])
    raw['T'] = (raw['temp_at_11m_front']+raw['temp_at_11m_back'])/2
    raw['day'] = (raw['Timestamp'].dt.date - REF_DATE).apply(lambda x: x.days)
    daily = raw.groupby('day').agg({'tiltx_11m':'mean','tiltx_19m':'mean','T':'mean'}).reset_index()
    daily = daily[daily['day']>=21].copy().reset_index(drop=True)
    daily['phase']=1; daily.loc[daily['day']>=46,'phase']=2
    daily.loc[daily['day']>=81,'phase']=3; daily.loc[daily['day']>=148,'phase']=4
    fea_sl = pd.read_csv(FEA_SLOPE_CSV); slope_cols = [c for c in fea_sl.columns if c.startswith('slope_')]
    daily = daily.merge(fea_sl[['day','net_udl_AB','net_udl_BC']+slope_cols], on='day', how='left')
    fea_df = pd.read_csv(FEA_DEFL_CSV); defl_cols = [c for c in fea_df.columns if c.startswith('defl_')]
    daily = daily.merge(fea_df[['day']+defl_cols], on='day', how='left')
    for ph in [1,2,3,4]: print(f"  Phase {ph}: {(daily['phase']==ph).sum()} days")
    return daily, slope_cols, defl_cols

## Run: Load Data and Train

Load all data, instantiate the CEI-PCNN solver, and run the full training pipeline (Adam 5000 epochs + L-BFGS 80 steps). Expected runtime: 3-5 minutes on CPU.

In [10]:
t0 = datetime.now()
print("="*70)
print("CEI-PCNN: phi = x^2*(x-15)^2*(x-30), PDE: w4+q/EI=0")
print("="*70)

daily, slope_cols, defl_cols = load_data()
N = len(daily); days = daily['day'].values; phases = daily['phase'].values
m11 = daily['tiltx_11m'].values; m19 = daily['tiltx_19m'].values; T = daily['T'].values
gpos = np.array([float(c.replace('slope_','').replace('m','')) for c in slope_cols])
fe_sl = daily[slope_cols].values; fe_df = daily[defl_cols].values
fe_q_AB = daily['net_udl_AB'].values; fe_q_BC = daily['net_udl_BC'].values

solver = CEI_PCNN(N, phases, T)
hist = solver.solve(m11, m19, n_adam=5000, n_lbfgs=80)

print(f"\n  Training time: {(datetime.now()-t0).total_seconds():.0f}s")

CEI-PCNN: phi = x^2*(x-15)^2*(x-30), PDE: w4+q/EI=0
  Phase 1: 25 days
  Phase 2: 35 days
  Phase 3: 67 days
  Phase 4: 126 days
    ep 0: Ld=7.531e-01 Lp=1.580e-08 Lc=2.797e-09 Lm=2.333e-05
    ep 1000: Ld=7.735e-03 Lp=1.417e-08 Lc=4.443e-09 Lm=3.219e-05
    ep 2000: Ld=5.893e-03 Lp=1.662e-08 Lc=4.065e-09 Lm=3.171e-05
    ep 3000: Ld=2.115e-03 Lp=1.599e-08 Lc=3.958e-09 Lm=3.080e-05
    ep 4000: Ld=4.283e-03 Lp=1.563e-08 Lc=3.992e-09 Lm=3.283e-05
    L-BFGS (80 steps)...
    L-BFGS 0
    L-BFGS 20
    L-BFGS 40
    L-BFGS 60

  Training time: 866s


## Evaluate Accuracy

Compute PCNN predictions at all spatial positions and compare against FEM at sensor positions and 4 validation positions (x = 5, 10, 20, 25 m). Also compare discovered loads q(t) against FEM net UDL.

In [11]:
pcnn_sl, pcnn_df = solver.predict_at(gpos)
ip11 = np.argmin(np.abs(gpos-11)); ip19 = np.argmin(np.abs(gpos-19))
pred_11 = pcnn_sl[:,ip11]; pred_19 = pcnn_sl[:,ip19]
m_s_AB = met(m11, pred_11); m_s_BC = met(m19, pred_19)
pq_AB = solver._q_AB.detach().numpy(); pq_BC = solver._q_BC.detach().numpy()
m_q_AB = met(fe_q_AB, pq_AB); m_q_BC = met(fe_q_BC, pq_BC)

print(f"\n{'='*70}\n(A) ACCURACY\n{'='*70}")
print(f"  Sensor: AB r={m_s_AB['r']:.4f} R2={m_s_AB['R2']:.4f} | BC r={m_s_BC['r']:.4f} R2={m_s_BC['R2']:.4f}")
vs = {}; vd = {}
for xv in VAL_POS:
    ip = np.argmin(np.abs(gpos-xv))
    ms = met(fe_sl[:,ip], pcnn_sl[:,ip]); md = met(fe_df[:,ip], pcnn_df[:,ip])
    vs[xv] = ms; vd[xv] = md
    print(f"  x={xv:.0f}m: sl r={ms['r']:.4f} R2={ms['R2']:.4f} Bias={ms['Bias']:+.3f} | "
          f"df r={md['r']:.4f} R2={md['R2']:.4f} Bias={md['Bias']:+.3f}")
mf_sl = met(fe_sl, pcnn_sl); mf_df = met(fe_df, pcnn_df)
print(f"  Load: AB r={m_q_AB['r']:.4f} RMSE={m_q_AB['RMSE']:.3f} | BC r={m_q_BC['r']:.4f} RMSE={m_q_BC['RMSE']:.3f}")


(A) ACCURACY
  Sensor: AB r=0.9919 R2=0.9837 | BC r=0.9857 R2=0.9706
  x=5m: sl r=0.9876 R2=0.8871 Bias=-0.050 | df r=0.9877 R2=0.7104 Bias=+0.391
  x=10m: sl r=0.9907 R2=0.7159 Bias=-0.088 | df r=0.9911 R2=0.8545 Bias=-0.321
  x=20m: sl r=0.9773 R2=-17.6163 Bias=+0.711 | df r=0.9822 R2=-4.9205 Bias=-1.647
  x=25m: sl r=0.3753 R2=-1283.3216 Bias=+2.781 | df r=0.9458 R2=-98.7378 Bias=+9.685
  Load: AB r=0.9008 RMSE=2.705 | BC r=0.8109 RMSE=4.276


## Evaluate Physics Compliance

Check boundary condition satisfaction (hard BCs should be ~1e-10), moment/shear continuity at the interior support, and PDE residual quality.

In [12]:
phys = evaluate_physics(solver, N)
print(f"\n{'='*70}\n(B) PHYSICS\n{'='*70}")
for k, v in phys['bc_violations_rms'].items():
    print(f"  {k:>10s}: {v:.2e}  [{'HARD' if v<1e-8 else 'SOFT'}]")
c = phys['continuity']
print(f"  Continuity: moment RMS={c['moment_rms']:.3e} | shear RMS={c['shear_rms']:.3e}")
for sn, p in phys['pde_residual'].items():
    print(f"  PDE {sn}: RMS={p['rms']:.3e} Max={p['max']:.3e}")

# ══════ PLOTS ══════


(B) PHYSICS
        w(0): 0.00e+00  [HARD]
       w(15): 0.00e+00  [HARD]
       w(30): 0.00e+00  [HARD]
       w'(0): 0.00e+00  [HARD]
      w'(15): 0.00e+00  [HARD]
     w''(30): 5.71e-03  [SOFT]
  Continuity: moment RMS=5.873e-05 | shear RMS=1.486e-05
  PDE AB: RMS=3.511e-05 Max=5.714e-05
  PDE BC: RMS=1.209e-04 Max=1.280e-04


## Plot: Training Convergence

All loss components on log scale. L_data should decrease steadily; physics losses decrease as their weights ramp up.

In [13]:
# 1. Convergence
fig, ax = plt.subplots(figsize=(12,5))
ax.semilogy(hist['data'],'b-',lw=0.5,alpha=0.7,label='L_data')
ax.semilogy(hist['pde'],'r-',lw=0.5,alpha=0.7,label='L_pde')
ax.semilogy(hist['cont'],'g-',lw=0.5,alpha=0.7,label='L_continuity')
ax.semilogy(hist['mom'],'m-',lw=0.5,alpha=0.7,label="L_moment w''(30)")
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('CEI-PCNN Convergence',fontsize=13,fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'01_convergence.png'),dpi=200); plt.close()

findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.


## Plot: Sensor Fit

Measured vs PCNN-reconstructed tiltx at sensor positions (x=11m and x=19m). Phase transitions marked with vertical dashed lines.

In [14]:
# 2. Sensor
fig, axes = plt.subplots(2,1,figsize=(14,8))
fig.suptitle('Sensor Fit',fontsize=13,fontweight='bold')
for ax,m,pred,ms,t in [(axes[0],m11,pred_11,m_s_AB,'AB x=11m'),(axes[1],m19,pred_19,m_s_BC,'BC x=19m')]:
    ax.plot(days,m,'b-',lw=1,alpha=0.7,label='Measured')
    ax.plot(days,pred,'r-',lw=1,alpha=0.7,label=f'PCNN (r={ms["r"]:.4f})')
    _vlines(ax); ax.set_ylabel('tiltx (mm/m)'); ax.set_title(t); ax.legend(); ax.grid(True,alpha=0.3)
axes[1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'02_sensor_fit.png'),dpi=200); plt.close()

## Plot: Validation Time Series

PCNN vs FEM at 4 uninstrumented validation positions. Left: slope, Right: deflection. These positions are NOT used during training.

In [15]:
# 3. Validation time series
fig, axes = plt.subplots(4,2,figsize=(16,16))
fig.suptitle('Validation: PCNN vs FEM',fontsize=13,fontweight='bold')
for row, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for col, fe, pc, ms, yl in [(0,fe_sl[:,ip],pcnn_sl[:,ip],vs[xv],'Slope (mm/m)'),
                                 (1,fe_df[:,ip],pcnn_df[:,ip],vd[xv],'Defl (mm)')]:
        axes[row,col].plot(days,fe,'b-',lw=1,alpha=0.7,label='FEM')
        axes[row,col].plot(days,pc,'r-',lw=1,alpha=0.7,label=f'PCNN (r={ms["r"]:.3f} Bias={ms["Bias"]:+.2f})')
        _vlines(axes[row,col]); axes[row,col].set_ylabel(yl)
        axes[row,col].set_title(f'x={xv:.0f}m'); axes[row,col].legend(fontsize=7); axes[row,col].grid(True,alpha=0.3)
axes[3,0].set_xlabel('Day'); axes[3,1].set_xlabel('Day')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'03_validation_ts.png'),dpi=200); plt.close()

## Plot: Validation Scatter (Phase-Coloured)

Scatter plots of PCNN vs FEM, coloured by loading phase. Points on the diagonal = perfect agreement.

In [16]:
# 4. Validation scatter
fig, axes = plt.subplots(2,4,figsize=(18,8))
fig.suptitle('Scatter: PCNN vs FEM',fontsize=13,fontweight='bold')
for col, xv in enumerate(VAL_POS):
    ip = np.argmin(np.abs(gpos-xv))
    for ph in [1,2,3,4]:
        mk = phases==ph
        axes[0,col].scatter(fe_sl[mk,ip],pcnn_sl[mk,ip],s=8,c=CP[ph],alpha=0.6,label=LP[ph] if col==0 else '')
        axes[1,col].scatter(fe_df[mk,ip],pcnn_df[mk,ip],s=8,c=CP[ph],alpha=0.6)
    for r in [0,1]:
        fe = fe_sl[:,ip] if r==0 else fe_df[:,ip]
        lm=[fe.min(),fe.max()]; axes[r,col].plot(lm,lm,'k--'); axes[r,col].grid(True,alpha=0.3)
    axes[0,col].set_title(f'x={xv:.0f}m Slope',fontsize=9)
    axes[1,col].set_title(f'x={xv:.0f}m Defl',fontsize=9); axes[1,col].set_xlabel('FEM')
axes[0,0].legend(fontsize=5,ncol=2); axes[0,0].set_ylabel('PCNN Slope'); axes[1,0].set_ylabel('PCNN Defl')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'04_validation_scatter.png'),dpi=200); plt.close()

## Plot: Mean Spatial Profiles

Mean slope and deflection profiles (averaged over all 253 days). Reveals systematic spatial bias from trial function shape mismatch.

In [17]:
# 5. Mean profiles
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Mean Profiles: FEM vs PCNN',fontsize=13,fontweight='bold')
for ax,fe,pc,yl in [(axes[0],fe_sl,pcnn_sl,'Slope (mm/m)'),(axes[1],fe_df,pcnn_df,'Defl (mm)')]:
    ax.plot(gpos,fe.mean(0),'k-',lw=2,label='FEM'); ax.plot(gpos,pc.mean(0),'r--',lw=2,label='CEI-PCNN')
    ax.axvline(11,color='blue',ls=':',lw=1.5,alpha=0.7); ax.axvline(19,color='blue',ls=':',lw=1.5,alpha=0.7)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.set_ylabel(yl); ax.legend(); ax.grid(True,alpha=0.3)
axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'05_mean_profiles.png'),dpi=200); plt.close()

## Plot: Load Discovery

PCNN-discovered load q(t) vs FEM net UDL. Good load discovery (high r) confirms the PDE loss is working correctly.

In [18]:
# 6. Load comparison
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Load: PCNN q(t) vs FEM',fontsize=13,fontweight='bold')
for col,sn,fq,pq,mq in [(0,'AB',fe_q_AB,pq_AB,m_q_AB),(1,'BC',fe_q_BC,pq_BC,m_q_BC)]:
    axes[0,col].plot(days,fq,'b-',lw=1,alpha=0.7,label='FEM')
    axes[0,col].plot(days,pq,'r-',lw=1,alpha=0.7,label=f'PCNN (r={mq["r"]:.3f})')
    for d in [46,81,148]: axes[0,col].axvline(d,color='gray',ls='--',lw=0.5)
    axes[0,col].set_ylabel('q (kN/m)'); axes[0,col].set_title(f'{sn} Load'); axes[0,col].legend(fontsize=7); axes[0,col].grid(True,alpha=0.3)
    for ph in [1,2,3,4]:
        mk=phases==ph; axes[1,col].scatter(fq[mk],pq[mk],s=12,c=CP[ph],alpha=0.6,label=LP[ph])
    lm=[min(fq.min(),pq.min()),max(fq.max(),pq.max())]; axes[1,col].plot(lm,lm,'k--')
    axes[1,col].set_xlabel('FEM (kN/m)'); axes[1,col].set_ylabel('PCNN (kN/m)')
    axes[1,col].set_title(f'{sn} RMSE={mq["RMSE"]:.2f}'); axes[1,col].legend(fontsize=6); axes[1,col].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'06_load.png'),dpi=200); plt.close()

## Plot: 3D Spatio-Temporal Surfaces

3D surface plots of slope(x,t) and deflection(x,t) showing the full reconstructed field over all days and positions.

In [19]:
# 7-8. 3D slope + defl
Xg,Tg = np.meshgrid(gpos,days)
for data,zl,fn in [(pcnn_sl,'Slope','07_3d_slope.png'),(pcnn_df,'Defl','08_3d_defl.png')]:
    fig=plt.figure(figsize=(16,10)); ax3=fig.add_subplot(111,projection='3d')
    ax3.plot_surface(Xg,Tg,data,cmap='RdBu_r',alpha=0.85,rstride=2,cstride=1,linewidth=0)
    ax3.set_xlabel('x (m)'); ax3.set_ylabel('Day'); ax3.set_zlabel(zl); ax3.view_init(25,-60)
    plt.tight_layout(); plt.savefig(os.path.join(OUT,fn),dpi=200); plt.close()

## Plot: Accuracy Metrics Bar Chart

Bar chart of Pearson r, RMSE, and Bias at each validation position, split by slope and deflection.

In [20]:
# 9. Accuracy bar
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Accuracy Metrics',fontsize=13,fontweight='bold')
xp = np.arange(len(VAL_POS)); wd = 0.35
for ax,metric,ylabel in [(axes[0],'r','Pearson r'),(axes[1],'RMSE','RMSE'),(axes[2],'Bias','Bias')]:
    sv=[vs[xv][metric] for xv in VAL_POS]; dv=[vd[xv][metric] for xv in VAL_POS]
    b1=ax.bar(xp-wd/2,sv,wd,label='Slope',color='steelblue',alpha=0.8)
    b2=ax.bar(xp+wd/2,dv,wd,label='Deflection',color='coral',alpha=0.8)
    for b,vals in [(b1,sv),(b2,dv)]:
        for bar,v in zip(b,vals):
            ax.annotate(f'{v:.3f}',xy=(bar.get_x()+bar.get_width()/2,bar.get_height()),
                        ha='center',va='bottom' if v>=0 else 'top',fontsize=7)
    ax.set_xticks(xp); ax.set_xticklabels([f'{xv:.0f}m' for xv in VAL_POS])
    ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
    if metric=='Bias': ax.axhline(0,color='k',lw=0.5)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'09_accuracy_bar.png'),dpi=200); plt.close()

## Plot: Physics Compliance

Three-panel chart: (1) BC violations on log scale, (2) continuity jumps (RMS/Max), (3) PDE residual per span.

In [21]:
# 10. Physics bar
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Physical Consistency',fontsize=13,fontweight='bold')
bc_n=list(phys['bc_violations_rms'].keys()); bc_v=[phys['bc_violations_rms'][k] for k in bc_n]
colors=['green' if v<1e-8 else 'orange' if v<0.01 else 'red' for v in bc_v]
axes[0].barh(bc_n,bc_v,color=colors,alpha=0.8); axes[0].set_xscale('log')
axes[0].set_xlabel('RMS'); axes[0].set_title('BC Compliance'); axes[0].grid(True,alpha=0.3)
cn=['Moment','Shear']; cr=[c['moment_rms'],c['shear_rms']]; cm=[c['moment_max'],c['shear_max']]
xp2=np.arange(2)
axes[1].bar(xp2-0.17,cr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[1].bar(xp2+0.17,cm,0.34,label='Max',color='coral',alpha=0.8)
axes[1].set_xticks(xp2); axes[1].set_xticklabels(cn); axes[1].set_title('Continuity x=15'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
pn=list(phys['pde_residual'].keys()); pr=[phys['pde_residual'][k]['rms'] for k in pn]; pm=[phys['pde_residual'][k]['max'] for k in pn]
xp3=np.arange(len(pn))
axes[2].bar(xp3-0.17,pr,0.34,label='RMS',color='steelblue',alpha=0.8)
axes[2].bar(xp3+0.17,pm,0.34,label='Max',color='coral',alpha=0.8)
axes[2].set_xticks(xp3); axes[2].set_xticklabels([f'Span {k}' for k in pn]); axes[2].set_title('PDE Residual'); axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'10_physics_bar.png'),dpi=200); plt.close()

## Plot: Daily Profiles

Selected daily profiles from different phases: FEM (solid) vs PCNN (dashed). Shows day-to-day variation, not just mean shape.

In [22]:
# 11. Daily profiles
target_days = [25, 85, 176, 268]
sel = [np.argmin(np.abs(days-td)) for td in target_days]
fig, axes = plt.subplots(2,1,figsize=(16,10))
fig.suptitle('Daily Profiles: FEM (solid) vs CEI-PCNN (dashed)',fontsize=13,fontweight='bold')
for j,di in enumerate(sel):
    ds = day_to_date(days[di]); cc = DC[j]
    axes[0].plot(gpos,fe_sl[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[0].plot(gpos,pcnn_sl[di],'--',color=cc,lw=2)
    axes[1].plot(gpos,fe_df[di],'-',color=cc,lw=2,label=f'{ds} (Ph{phases[di]})')
    axes[1].plot(gpos,pcnn_df[di],'--',color=cc,lw=2)
axes[0].plot([],[],'-k',lw=2,label='FEM'); axes[0].plot([],[],'--k',lw=2,label='PCNN')
for ax in axes:
    ax.axvline(11,color='blue',ls=':',lw=1,alpha=0.5); ax.axvline(19,color='red',ls=':',lw=1,alpha=0.5)
    ax.axvline(15,color='black',lw=2,alpha=0.3); ax.legend(fontsize=8,ncol=3); ax.grid(True,alpha=0.3)
axes[0].set_ylabel('Slope (mm/m)'); axes[1].set_ylabel('Deflection (mm)'); axes[1].set_xlabel('x (m)')
plt.tight_layout(); plt.savefig(os.path.join(OUT,'11_daily_profiles.png'),dpi=200); plt.close()

## Save Results

Save pcnn_results.csv (all predictions at all positions) and pcnn_summary.json (accuracy + physics metrics).

In [23]:
# Save
df_out = pd.DataFrame({'day':days,'phase':phases,'tiltx_11m':m11,'tiltx_19m':m19,
    'pcnn_pred_11m':pred_11,'pcnn_pred_19m':pred_19,'q_AB':pq_AB,'q_BC':pq_BC,'fe_q_AB':fe_q_AB,'fe_q_BC':fe_q_BC})
for ip,xv in enumerate(gpos):
    df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
df_out.to_csv(os.path.join(OUT,'pcnn_results.csv'),index=False)

with open(os.path.join(OUT,'pcnn_summary.json'),'w') as f:
    json.dump({'method':'CEI-PCNN (phi=x^2(x-15)^2(x-30), PDE w4+q/EI=0, continuity+moment)',
        'EI':float(EI_NOM),'trial_function':'x^2*(x-15)^2*(x-30)',
        'accuracy':{'sensor':{'AB':m_s_AB,'BC':m_s_BC},
            'validation':{f'x{xv:.0f}m':{'slope':vs[xv],'defl':vd[xv]} for xv in VAL_POS},
            'fullfield':{'slope':mf_sl,'defl':mf_df},'load':{'AB':m_q_AB,'BC':m_q_BC}},
        'physics':phys},f,indent=2,default=str)

print(f"\n  11 plots + CSV + JSON saved to {OUT}/")
print(f"  Runtime: {(datetime.now()-t0).total_seconds():.0f}s")


  11 plots + CSV + JSON saved to C:\Users\ridoc\OneDrive\Desktop\Folders\SMACCs\Thesis\Physics Guided Framework\Draft\Models\outputs\processing\Tiltx\Single_Trial_function_PCNN/
  Runtime: 987s


C:\Users\ridoc\AppData\Local\Temp\ipykernel_26732\1894083655.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_26732\1894083655.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'pcnn_slope_{xv:.1f}m']=pcnn_sl[:,ip]; df_out[f'pcnn_defl_{xv:.1f}m']=pcnn_df[:,ip]
C:\Users\ridoc\AppData\Local\Temp\ipykernel_26732\1894083655.py:5: PerformanceWarning: DataFrame is highly fragmented.  This